# Exercise: Implement the GPT Model Configuration

Welcome! In this exercise, you'll get hands-on experience with a fundamental part of building any transformer model: the configuration. A configuration object is a clean way to store all the key hyperparameters that define a model's architecture.

By the end of this exercise, you will be able to:
*   Define a Python `dataclass` to hold model hyperparameters.
*   Instantiate this class to create configurations for different model sizes.
*   Use your configuration class in a helper "factory" function.

Let's get started!

In [ ]:
# 1. Setup
# You don't need to modify this cell.

import math
from dataclasses import dataclass
from typing import Dict

# Helper function for the optional challenge.
def count_parameters(config: "GPTConfig") -> int:
    """
    A simplified function to estimate the number of parameters in a GPT model.
    This is a rough approximation based on the nanoGPT repository's formula.
    """
    # token + positional embeddings
    token_embed = config.vocab_size * config.n_embd
    pos_embed = config.block_size * config.n_embd
    
    # transformer blocks
    # simplified rule of thumb: 12 * n_embd^2 per block
    # (4*n_embd^2 for attn, and 8*n_embd^2 for MLP)
    block_params = config.n_layer * (12 * config.n_embd**2)
    
    # final layer norm
    final_ln = config.n_embd
    
    # the output head is often tied to the token embedding, but we count it here
    lm_head = config.vocab_size * config.n_embd
    
    # Note: This formula omits biases and some layer norm parameters for simplicity.
    total = token_embed + pos_embed + block_params + final_ln + lm_head
    return total

## Exercise 1: Implement the `GPTConfig` Dataclass

Your first task is to define the configuration for our GPT model. We'll use Python's `@dataclass` to create a simple, clean class for holding our hyperparameters. A `dataclass` automatically generates methods like `__init__` and `__repr__` for you, which is very convenient!

**Instructions:**
You will need to define the following attributes inside the class, along with their type hints and default values:

*   `block_size`: The maximum context length for the model. (Default: `1024`)
*   `vocab_size`: The number of tokens in the vocabulary. (Default: `50257`, the size for GPT-2)
*   `n_layer`: The number of transformer blocks (layers) in the model. (Default: `12`)
*   `n_head`: The number of attention heads in each transformer block. (Default: `12`)
*   `n_embd`: The dimensionality of the token embeddings. (Default: `768`)

**Hint:**
Remember to use the `@dataclass` decorator right above your class definition. Inside the class, define each attribute on its own line like this: `attribute_name: type = default_value`.

In [ ]:
# GRADED CLASS: GPTConfig

@dataclass
class GPTConfig:
    """Configuration for a GPT model."""
    ### START CODE HERE ###
    block_size: int = 1024
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    ### END CODE HERE ###

In [ ]:
# Test cell for Exercise 1

# Test default instantiation
print("Testing default configuration...")
default_config = GPTConfig()
assert default_config.block_size == 1024, f"Default block_size is incorrect. Got {default_config.block_size}, expected 1024"
assert default_config.vocab_size == 50257, f"Default vocab_size is incorrect. Got {default_config.vocab_size}, expected 50257"
assert default_config.n_layer == 12, f"Default n_layer is incorrect. Got {default_config.n_layer}, expected 12"
assert default_config.n_head == 12, f"Default n_head is incorrect. Got {default_config.n_head}, expected 12"
assert default_config.n_embd == 768, f"Default n_embd is incorrect. Got {default_config.n_embd}, expected 768"
print("✅ Default configuration tests passed!")

# Test custom instantiation
print("\nTesting custom configuration...")
custom_config = GPTConfig(n_layer=6, n_embd=384, block_size=256)
assert custom_config.n_layer == 6, f"Custom n_layer is incorrect. Got {custom_config.n_layer}, expected 6"
assert custom_config.n_embd == 384, f"Custom n_embd is incorrect. Got {custom_config.n_embd}, expected 384"
assert custom_config.block_size == 256, f"Custom block_size is incorrect. Got {custom_config.block_size}, expected 256"
assert custom_config.vocab_size == 50257, "A non-overridden default value was changed!"
print("✅ Custom configuration tests passed!")

print("\n🎉 All tests for Exercise 1 passed!")

## Exercise 2: Create a Factory Function for nanoGPT

Great job! Now that you have a flexible configuration class, let's create a "factory" function. This is a common pattern where a function returns a pre-configured object, making it easy to create standard model sizes.

**Instructions:**
Your goal is to implement `create_nano_gpt_config()`. This function should return a `GPTConfig` instance with the specific dimensions used in Andrej Karpathy's popular 'nanoGPT' project, which is a great small model for training from scratch.

The required specifications for nanoGPT are:
*   `n_layer`: 6
*   `n_head`: 6
*   `n_embd`: 384
*   `block_size`: 256

All other parameters should keep their default values from the `GPTConfig` class.

In [ ]:
# GRADED FUNCTION: create_nano_gpt_config

def create_nano_gpt_config() -> GPTConfig:
    """
    Returns a GPTConfig instance with the nanoGPT model dimensions.
    """
    ### START CODE HERE ###
    return GPTConfig(
        block_size=256,
        n_layer=6,
        n_head=6,
        n_embd=384
    )
    ### END CODE HERE ###

In [ ]:
# Test cell for Exercise 2

print("Testing create_nano_gpt_config()...")
nano_config = create_nano_gpt_config()

# Test the type of the returned object
assert isinstance(nano_config, GPTConfig), f"Function should return a GPTConfig object, but got {type(nano_config)}"

# Test the specific nanoGPT values
assert nano_config.n_layer == 6, f"nanoGPT n_layer is incorrect. Got {nano_config.n_layer}, expected 6"
assert nano_config.n_head == 6, f"nanoGPT n_head is incorrect. Got {nano_config.n_head}, expected 6"
assert nano_config.n_embd == 384, f"nanoGPT n_embd is incorrect. Got {nano_config.n_embd}, expected 384"
assert nano_config.block_size == 256, f"nanoGPT block_size is incorrect. Got {nano_config.block_size}, expected 256"

# Test that other values remain default
assert nano_config.vocab_size == 50257, f"nanoGPT vocab_size should be the default. Got {nano_config.vocab_size}, expected 50257"

print("\n🎉 All tests for Exercise 2 passed!")

## (Optional) Challenge: Compare Model Sizes

Excellent work! As an optional challenge, let's see how these configuration parameters directly impact the total size of the model. A larger model can have a higher capacity to learn, but it also requires more computational resources and data to train effectively.

We've provided a helper function `count_parameters` that gives a rough estimate of the number of parameters in a model based on a `GPTConfig` object. We've also provided a dictionary `gpt2_configs` containing configurations for standard GPT-2 model sizes.

**Your Task:**
Implement the `print_config_summary` function. This function should iterate through the dictionary of configs and print a formatted summary showing each model's name, key dimensions, and estimated parameter count in millions (M).

Example output format:
```
--------------------------------------------------------------------------
Model Name      | Layers | Embedding Dim | Heads | Parameters (M)
--------------------------------------------------------------------------
nanoGPT         | 6      | 384           | 6     | 10.7
gpt2            | 12     | 768           | 12    | 124.4
... (and so on for the other models)
--------------------------------------------------------------------------
```
This is a great way to see the relationship between `n_layer`, `n_embd`, and the explosive growth in model size!

In [ ]:
# Setup for the optional challenge
# You don't need to modify this dictionary.

gpt2_configs: Dict[str, GPTConfig] = {
    # Our nanoGPT config from Exercise 2
    "nanoGPT": create_nano_gpt_config(),
    # The default config from Exercise 1, which matches GPT-2 "small" (124M)
    "gpt2": GPTConfig(),
    # GPT-2 "medium" (355M)
    "gpt2-medium": GPTConfig(n_layer=24, n_head=16, n_embd=1024),
    # GPT-2 "large" (774M)
    "gpt2-large": GPTConfig(n_layer=36, n_head=20, n_embd=1280),
}


def print_config_summary(configs: Dict[str, GPTConfig]):
    """
    Prints a summary table of model configurations and their parameter counts.
    
    Args:
        configs: A dictionary where keys are model names and values are GPTConfig objects.
    """
    # Print table header
    print("-" * 74)
    print(f"{'Model Name':<15} | {'Layers':<6} | {'Embedding Dim':<13} | {'Heads':<5} | {'Parameters (M)':<15}")
    print("-" * 74)
    
    ### START CODE HERE ###
    # Iterate through the configs dictionary
    for name, config in configs.items():
        # Calculate the number of parameters
        params = count_parameters(config)
        
        # Convert parameters to millions and format to one decimal place
        params_m = f"{params / 1e6:.1f}"
        
        # Print a formatted row for the current model
        print(f"{name:<15} | {config.n_layer:<6} | {config.n_embd:<13} | {config.n_head:<5} | {params_m:<15}")
    ### END CODE HERE ###

    # Print table footer
    print("-" * 74)


# --- Run the summary function ---
# Visually inspect the output to see if it matches the expected format.
print_config_summary(gpt2_configs)